<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/LiDARtrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

#import tensorflow as tf

from pyquaternion import Quaternion

TORCH_version = torch.__version__.split('+')[0]
#CUDA_version = torch.version.cuda.replace('.', '')

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

from scipy.ndimage import maximum_filter

import sys

#!pip install import-ipynb

!pip install open3d plotly

import open3d as o3d
import matplotlib.pyplot as plt

#import import_ipynb

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!npx degit google-research-datasets/Objectron/objectron objectron

sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box

⠙⠹⠸⠼⠴Need to install the following packages:
degit@3.6.1
Ok to proceed? (y) y

⠙⠹⠸> cloned google-research-datasets/Objectron#HEAD to objectron
⠙

In [3]:
!pip install nuscenes-devkit &> /dev/null  # Install nuScenes.

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, Box
from nuscenes.eval.detection.utils import category_to_detection_name

nusc_root = "/content/drive/MyDrive/LiDARFusion/nuscenes/datanuscenes"

nusc = NuScenes(version='v1.0-mini', dataroot=nusc_root, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 7.446 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


In [45]:
sys.path.insert(0, '/content/drive/MyDrive/LiDARFusion')
%cd /content/drive/MyDrive/LiDARFusion

from voxel_pointnet2 import PointNetSetAbstraction

from centerpoint import CenterPoint

from lidar_baseline import PillarsEncoder, get_ij, get_point_pillar

#import CenterPointHead

/content/drive/MyDrive/LiDARFusion


features(Δt, intensity) are kept in separate matrices, and pasted together for encoding

In [5]:
test_scene = nusc.scene[0]

token_0 = test_scene["first_sample_token"]

cloud_sample = nusc.get("sample", token_0)
#cloud_data = nusc.get("sample_data", cloud_sample["data"]["LIDAR_TOP"])

#cloud_root = os.path.join(nusc_root, cloud_data["filename"])

#cloud = LidarPointCloud.from_file(cloud_root)

#cloud_feats = cloud.points[3:,:].T

In [ ]:
def extract_feats(scene, data_root):

  scene_tokens = []
  token_0 = scene["first_sample_token"]

  while token_0 != "":
    scene_tokens.append(token_0)

    token_sample = nusc.get("sample", token_0)
    token_0 = token_sample["next"]

  scene_feats = []

  for i in range(len(scene_tokens)):
    cloud_sample = nusc.get("sample", scene_tokens[i])
    cloud_data = nusc.get("sample_data", cloud_sample["data"]["LIDAR_TOP"])

    cloud_root = os.path.join(data_root, cloud_data["filename"])

    cloud = LidarPointCloud.from_file(cloud_root)

    cloud_feats = cloud.points[3:, :].T

    scene_feats.append(cloud_feats)

  return scene_feats

In [27]:
def load_sweeps(nusc, sample, N=10,min_distance = 1.0):

  pc, times = LidarPointCloud.from_file_multisweep(nusc=nusc, sample_rec = sample, chan = "LIDAR_TOP", ref_chan = "LIDAR_TOP", nsweeps = N, min_distance = min_distance)

  out = np.concatenate([pc.points, times], axis =0).T

  return out

In [43]:
def get_grid_attr(cloud, voxel_size = 1):

  min_x = np.min(cloud[:, 0:1])
  min_y = np.min(cloud[:, 1:2])
  min_z = np.min(cloud[:, 2:3])

  min_bound = np.array([min_x,min_y,min_z])

  max_x = np.max(cloud[:, 0:1])
  max_y = np.max(cloud[:, 1:2])
  max_z = np.max(cloud[:, 2:3])

  max_bound = np.array([max_x, max_y, max_z])

  grid_shape = np.ceil((max_bound - min_bound) / voxel_size).astype(int)

  return min_bound, max_bound, grid_shape


In [46]:
def get_pillars(cloud, voxel_size = 1, pillar_max = 100): #voxel_size is length of one side of voxel

  min_bound, max_bound, grid_shape = get_grid_attr(cloud, voxel_size=voxel_size)

  T = grid_shape[0]*grid_shape[1]

  N_counter = np.zeros(T)

  for k in range(cloud.shape[0]):
    point = cloud[k]

    p_index = get_point_pillar(point, min_bound, voxel_size, grid_shape[0])

    N_counter[p_index] += 1

  N = (np.max(N_counter)).astype(int)

  if pillar_max < N or pillar_max == None:
    N = pillar_max

  out = np.zeros((T, N, 5))

  mask = np.zeros((T), dtype = int)

  for m in range(cloud.shape[0]):
    point = cloud[m]

    p_index = get_point_pillar(point, min_bound, voxel_size, grid_shape[0])

    k = mask[p_index]

    if k < N:
      out[p_index][k] = point

      mask[p_index] += 1

  out_zeros = ~np.all(out == 0, axis=(1, 2))

  out = out[out_zeros,:,:]

  mask = mask[mask != 0] #mask returns how many non-padded points are in each pillar. Zero points in a pillar are assumed to be padded unless the mask indicates otherwise

  out = torch.from_numpy(out).float()
  mask = torch.from_numpy(mask).float()

  return out, mask



In [ ]:
out = load_sweeps(nusc, cloud_sample)

In [48]:
pillars, mask = get_pillars(out)

In [50]:
pillars.shape

torch.Size([2314, 100, 5])

In [123]:
boxes = nusc.get_boxes(cloud_sample["data"]['LIDAR_TOP'])

box.corners() gives vertices

In [122]:
def boxes_to_lidar(nusc, sample, boxes):

  ego_pose = nusc.get("ego_pose", sample["data"]["LIDAR_TOP"])

  sd_rec = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
  cal_sensor = nusc.get('calibrated_sensor', sd_rec['calibrated_sensor_token'])

  out_boxes = []

  for i in range(len(boxes)):
    box = boxes[i]
    category_name = box.name

    detection_name = category_to_detection_name(category_name)

    box.translate(-np.array(ego_pose['translation']))
    box.rotate(Quaternion(ego_pose['rotation']).inverse)

    box.translate(-np.array(cal_sensor['translation']))
    box.rotate(Quaternion(cal_sensor['rotation']).inverse)

    box.name = detection_name

    out_boxes.append(box)

  return out_boxes


In [124]:
lidar_boxes = boxes_to_lidar(nusc, cloud_sample, boxes)